In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

hf_token = "HF_TOKEN"
model_name = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
model.eval()
print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [ ]:
def generate_response(user_prompt, system_prompt=None, max_new_tokens=200,
                      do_sample=False, temperature=1.0, top_p=1.0, top_k=50):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

# Sanity check
print(generate_response("What is the capital of France? One sentence."))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The capital of France is Paris.


In [ ]:
from datasets import load_dataset

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(dolly[0])
print("\nFields:", dolly.column_names)
print("Total entries:", len(dolly))

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}

Fields: ['instruction', 'context', 'response', 'category']
Total entries: 15011


In [ ]:
import random
random.seed(42)

# Closed-book entries (empty context) give a cleaner stylistic comparison
closed_book = [d for d in dolly if d['context'].strip() == '']
print(f"Closed-book entries: {len(closed_book)}")

samples = random.sample(closed_book, 100)
print(f"Sampled {len(samples)}")
print("\nExample prompt:", samples[0]['instruction'])
print("Example human answer:", samples[0]['response'][:200])

Closed-book entries: 10544
Sampled 100

Example prompt: Identify which instrument is string or percussion: Tan-tan, Ruan
Example human answer: Ruan is string, Tan-tan is percussion.


In [ ]:
import json

pairs = []
for i, d in enumerate(samples):
    ai_response = generate_response(
        d['instruction'],
        max_new_tokens=200,
        do_sample=True, temperature=0.7, top_p=0.9
    )
    pairs.append({
        "prompt": d['instruction'],
        "human": d['response'],
        "ai": ai_response,
    })
    with open("detection_pairs.json", "w") as f:
        json.dump(pairs, f, indent=2)        # save every step
    if i % 5 == 0:
        print(f"{i+1}/100 | {d['instruction'][:50]}")

print(f"\nDone. {len(pairs)} pairs.")

1/100 | Identify which instrument is string or percussion:
6/100 | In what industries is GIS indispensable?
11/100 | What is a credit score?
16/100 | What is a flat earther and is it possible for the 
21/100 | What do you need for car camping?
26/100 | In the series A Song of Ice and Fire, who is the f
31/100 | Identify from the following list all of the ingred
36/100 | Which team's have the most NCAA Division I men's b
41/100 | When did google first start?
46/100 | Why are mama's boys the worst partners?
51/100 | How long does it take our sun to orbit the center 
56/100 | When running a marathon and attempting to run a pe
61/100 | Which of these are movies that Will Ferrell starre
66/100 | Can you brainstorm some ideas for blog posts about
71/100 | Classify the below based on the mode of transporta
76/100 | Identify which animal species is alive or extinct:
81/100 | Tell me whether these are primary or secondary col
86/100 | Why do some countries call football soccer?
91/100 | Tell me

In [ ]:
# Code for recovery, in case the colab breaks in between
import json
with open("detection_pairs.json") as f:
    pairs = json.load(f)
print(f"Recovered {len(pairs)}. Resuming from {len(pairs)}.")

for i in range(len(pairs), 100):
    d = samples[i]
    ai_response = generate_response(d['instruction'], max_new_tokens=200,
                                    do_sample=True, temperature=0.7, top_p=0.9)
    pairs.append({"prompt": d['instruction'], "human": d['response'], "ai": ai_response})
    with open("detection_pairs.json", "w") as f:
        json.dump(pairs, f, indent=2)
    if i % 5 == 0:
        print(f"{i+1}/100")
print("Recovery complete.")

Recovered 100. Resuming from 100.
Recovery complete.


In [ ]:
from collections import Counter
import re, math

def get_ngrams(text, n):
    words = re.findall(r"\b\w+\b", text.lower())
    return [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]

def corpus_ngrams(texts, n):
    c = Counter()
    for t in texts:
        c.update(get_ngrams(t, n))
    return c

human_texts = [p["human"] for p in pairs]
ai_texts    = [p["ai"] for p in pairs]

ngram_results = {}

for n in [1, 2, 3, 4]:
    human_ng = corpus_ngrams(human_texts, n)
    ai_ng    = corpus_ngrams(ai_texts, n)
    total_human = sum(human_ng.values()) or 1
    total_ai    = sum(ai_ng.values()) or 1
    V = len(set(human_ng) | set(ai_ng))   # vocab size for smoothing

    scored = []
    for ng, ai_count in ai_ng.items():
        # Require the phrase to appear across MULTIPLE prompts, not one long answer
        n_docs = sum(1 for t in ai_texts if " ".join(ng) in t.lower())
        if ai_count < 5 or n_docs < 3:     # must be frequent AND spread out
            continue
        human_count = human_ng.get(ng, 0)
        # Add-one (Laplace) smoothing — no division-by-zero explosions
        ai_freq    = (ai_count + 1) / (total_ai + V)
        human_freq = (human_count + 1) / (total_human + V)
        log_ratio  = math.log(ai_freq / human_freq)
        scored.append((" ".join(ng), ai_count, human_count, n_docs, log_ratio))

    scored.sort(key=lambda x: x[4], reverse=True)
    ngram_results[n] = scored

    print(f"\n=== Top 15 AI-favored {n}-grams (n>=3 prompts) ===")
    print(f"{'phrase':35s} {'AI':>4s} {'Hum':>4s} {'#prompts':>8s} {'log-ratio':>9s}")
    for phrase, aic, humc, ndoc, lr in scored[:15]:
        print(f"{phrase:35s} {aic:4d} {humc:4d} {ndoc:8d} {lr:9.2f}")


=== Top 15 AI-favored 1-grams (n>=3 prompts) ===
phrase                                AI  Hum #prompts log-ratio
instrument                            25    0        7      2.74
known                                 24    0       20      2.70
key                                   20    0       21      2.53
especially                            19    0       18      2.48
r                                     14    0      100      2.19
series                                14    0        5      2.19
why                                   13    0       11      2.12
often                                 26    1       19      2.09
e                                     25    1      100      2.05
g                                     24    1       99      2.01
traditional                           11    0       10      1.97
major                                 11    0        9      1.97
fun                                   11    0       12      1.97
specifically                          11

In [ ]:
def find_examples(phrase, texts, max_examples=2):
    out = []
    for t in texts:
        if phrase in t.lower():
            for sent in re.split(r'(?<=[.!?])\s+', t):
                if phrase in sent.lower():
                    out.append(sent.strip())
                    if len(out) >= max_examples:
                        return out
    return out

print("=== Examples for top AI 3-grams ===\n")
for phrase, aic, humc, ndoc, lr in ngram_results[3][:8]:
    print(f"'{phrase}'  (AI:{aic} Human:{humc} #prompts:{ndoc} log-ratio:{lr:.2f})")
    for ex in find_examples(phrase, ai_texts, 2):
        print(f"  → {ex[:200]}")
    print()

=== Examples for top AI 3-grams ===

'is not a'  (AI:11 Human:0 #prompts:10 log-ratio:2.22)
  → House Qoherys is not a house in the *A Song of Ice and Fire* series.
  → However, even House Qohari is not a major house and does not have a well-documented founder in the canon of George R.R.

'a type of'  (AI:10 Human:0 #prompts:7 log-ratio:2.13)
  → - It is a **percussion instrument**, typically a type of drum or frame drum.
  → - **Pike** is a type of fish, specifically a predatory freshwater fish in the family *Esocidae*.

'a song of'  (AI:8 Human:0 #prompts:4 log-ratio:1.93)
  → House Qoherys is not a house in the *A Song of Ice and Fire* series.
  → House Seaworth is not a house in the *A Song of Ice and Fire* series.

'song of ice'  (AI:8 Human:0 #prompts:4 log-ratio:1.93)
  → House Qoherys is not a house in the *A Song of Ice and Fire* series.
  → House Seaworth is not a house in the *A Song of Ice and Fire* series.

'of ice and'  (AI:8 Human:0 #prompts:4 log-ratio:1.93)
  → House Q

In [ ]:
import statistics

def avg_words(texts):
    return statistics.mean(len(re.findall(r"\b\w+\b", t)) for t in texts)

print(f"Avg AI length:    {avg_words(ai_texts):.1f} words")
print(f"Avg human length: {avg_words(human_texts):.1f} words")

print("\nStrongest tell by n-gram size:")
for n in [1, 2, 3, 4]:
    if ngram_results[n]:
        phrase, aic, humc, ndoc, lr = ngram_results[n][0]
        print(f"  {n}-gram: '{phrase}' (log-ratio {lr:.2f}, AI {aic} vs Human {humc}, in {ndoc} prompts)")

Avg AI length:    120.1 words
Avg human length: 55.2 words

Strongest tell by n-gram size:
  1-gram: 'instrument' (log-ratio 2.74, AI 25 vs Human 0, in 7 prompts)
  2-gram: 'or a' (log-ratio 2.74, AI 20 vs Human 0, in 35 prompts)
  3-gram: 'is not a' (log-ratio 2.22, AI 11 vs Human 0, in 10 prompts)
  4-gram: 'a song of ice' (log-ratio 1.94, AI 8 vs Human 0, in 4 prompts)


In [ ]:
# pip install sentence-transformers   (run once)
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')  # small, fast, fine for this

human_texts = [p["human"] for p in pairs]
ai_texts    = [p["ai"]    for p in pairs]

h_emb = embedder.encode(human_texts, normalize_embeddings=True, show_progress_bar=True)
a_emb = embedder.encode(ai_texts,    normalize_embeddings=True, show_progress_bar=True)

# Within-pair cosine similarity (semantic closeness of human vs model answer, same prompt)
within_pair_sim = np.sum(h_emb * a_emb, axis=1)   # both already normalized

print(f"Mean within-pair semantic similarity: {within_pair_sim.mean():.3f}")
print(f"Median: {np.median(within_pair_sim):.3f}")
print(f"Std:    {within_pair_sim.std():.3f}")
print(f"Pairs with sim > 0.6 (semantically aligned): {(within_pair_sim > 0.6).sum()}/100")
print(f"Pairs with sim < 0.3 (topically divergent):  {(within_pair_sim < 0.3).sum()}/100")

# The most topically divergent pairs — inspect these for the writeup
order = np.argsort(within_pair_sim)
print("\n--- 5 most topically divergent pairs (lowest similarity) ---")
for i in order[:5]:
    print(f"\n[sim={within_pair_sim[i]:.2f}] PROMPT: {pairs[i]['prompt'][:80]}")
    print(f"  HUMAN: {pairs[i]['human'][:150]}")
    print(f"  AI:    {pairs[i]['ai'][:150]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Mean within-pair semantic similarity: 0.677
Median: 0.735
Std:    0.185
Pairs with sim > 0.6 (semantically aligned): 72/100
Pairs with sim < 0.3 (topically divergent):  5/100

--- 5 most topically divergent pairs (lowest similarity) ---

[sim=0.04] PROMPT: are GSIs important to being a global company
  HUMAN: yes
  AI:    Yes, **Global Service Integrators (GSIs)** are important to being a global company — but it's important to clarify what "GSI" means in this context, a

[sim=0.14] PROMPT: Danny Kaye Humanitarian Award 2019, was given to?
  HUMAN: Priyanka Chopra, an Indian Actress
  AI:    The Danny Kaye Humanitarian Award in 2019 was given to **Randy Johnston**.

However, it's important to note that the Danny Kaye Humanitarian Award is 

[sim=0.21] PROMPT: Write a creative ending scene for a thriller movie
  HUMAN: After 18 months of finding the treasure, the people of Corinth gathered to praise Big B and his friends for their achievement. Amidst the celebrations
  AI:    The rain ha